# AI Engagement Sortition Selection

This notebook runs the sortition algorithm to select a representative panel from survey respondents. It:
1. Fetches stratification targets and candidate data from Snowflake
2. Runs the sortition algorithm with configured constraints
3. Writes the selected panel back to Snowflake

In [ ]:
# Sortition settings
number_people_wanted = 150
allowed_deviation = 0.50  # how much deviation from the target distribution we allow
id_column = "SURVEY_RESPONDENT_ID"
columns_to_keep = []  # additional columns to keep in the output
selection_algorithm = "maximin"  # default is maximin

# Snowflake target configuration
TARGET_DATABASE = "TRANSFORM_ENGCA_PRD"
TARGET_SCHEMA = "AI_ENGAGEMENT"
TARGET_TABLE = "INT_AI_ENGAGEMENT_SORTITION_SELECTIONS"

In [ ]:
%%sql -r targets_df
SELECT
    question as category,
    answer as name,
    adjusted_target_pct * ({{number_people_wanted}} + 88) as target
FROM TRANSFORM_ENGCA_PRD.GOVOCAL.INT_GOVOCAL_SORTITION_TARGETS

Next we want to get the people who are candidates for the sortition run. We only want to include candidates who have not already been selected, so we filter out any candidates from previous sortition runs.

In [ ]:
%%sql -r people_df
SELECT *
FROM TRANSFORM_ENGCA_PRD.GOVOCAL.INT_GOVOCAL_SORTITION_CANDIDATES
WHERE 
    survey_respondent_id NOT IN (
        SELECT DISTINCT survey_respondent_id FROM TRANSFORM_ENGCA_PRD.AI_ENGAGEMENT.INT_AI_ENGAGEMENT_SORTITION_SELECTIONS)
    and lower(invitee_status) <> 'accepted'

In [ ]:
%%sql -r candidate_demo_count_df
SELECT
    question as category,
    answer as name,
    count(*) as candidate_count
FROM {{people_df}}
UNPIVOT (
    answer for question in (age, gender_category, race_ethnicity_category, region, field_of_work, ai_response_label)
)
GROUP BY question, answer

In [ ]:
%%sql -r accepted_demo_count_df
SELECT
    question as category,
    answer as name,
    count(*) as accepted_count
FROM TRANSFORM_ENGCA_PRD.GOVOCAL.INT_GOVOCAL_SORTITION_CANDIDATES
UNPIVOT (
    answer for question in (age, gender_category, race_ethnicity_category, region, field_of_work, ai_response_label)
)
WHERE 
    invitee_status = 'accepted'
GROUP BY question, answer


In [ ]:
import pandas as pd

features_df = targets_df.merge(candidate_demo_count_df, on=["CATEGORY", "NAME"], how="left",)
features_df["CANDIDATE_COUNT"] = features_df["CANDIDATE_COUNT"].fillna(0).astype(int)

# Adjust features_df min/max by subtracting already-accepted candidates
features_df = features_df.merge(accepted_demo_count_df, on=["CATEGORY", "NAME"], how="left",)
features_df["ACCEPTED_COUNT"] = features_df["ACCEPTED_COUNT"].fillna(0).astype(int)
features_df["REMAINING_TARGET"] = features_df["TARGET"] - features_df["ACCEPTED_COUNT"]

# features_df["MAX"] = features_df["REMAINING_TARGET"] * ( 1 + allowed_deviation)
features_df['MAX'] = features_df.apply(lambda row: max(row['REMAINING_TARGET'] * ( 1 + allowed_deviation + .50), 10) if row['CATEGORY'] == 'FIELD_OF_WORK' else row['REMAINING_TARGET'] * ( 1 + allowed_deviation), axis=1)
features_df['MIN'] = features_df.apply(lambda row: row['REMAINING_TARGET'] * ( 1 - allowed_deviation - .50) if row['CATEGORY'] == 'FIELD_OF_WORK' else row['REMAINING_TARGET'] * ( 1 - allowed_deviation), axis=1)



features_df["MIN"] = features_df["MIN"].round(0).astype(int)
features_df["MAX"] = features_df["MAX"].round(0).astype(int)

# MAX: floor of 0
features_df["MAX"] = features_df["MAX"].clip(lower=0)

# MIN: 0 if MAX is 0, otherwise min should not exceed CANDIDATE_COUNT, otherwise clip to 0 for Non-response, clip to 1 for all other category values
features_df["MIN"] = features_df.apply(
    lambda row: 0 if row["MAX"] == 0
    else max(0, min(row["CANDIDATE_COUNT"], row["MIN"])) if row["NAME"] == "Non-response" 
    else max(1, min(row["CANDIDATE_COUNT"], row["MIN"])),
    axis=1,
)
features_df = features_df.drop(columns=["ACCEPTED_COUNT", "CANDIDATE_COUNT"])

features_df

In [ ]:
import os
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql("USE ROLE TRANSFORM_ENGCA_PRD_READWRITECONTROL").collect()

# List all .whl files in the stage
files = session.sql("LIST @TRANSFORM_ENGCA_PRD.UTILITIES.PYTHON_WHEELS PATTERN='.*\\.whl'").collect()

# Download each file individually
os.makedirs('/tmp/wheels', exist_ok=True)
for f in files:
    filename = f['name'].split('/')[-1]
    session.file.get(f"@TRANSFORM_ENGCA_PRD.UTILITIES.PYTHON_WHEELS/{filename}", '/tmp/wheels')

# Install all downloaded wheels
!pip install /tmp/wheels/*.whl --quiet

from sortition_algorithms import (
    run_stratification,
    read_in_features,
    read_in_people,
    Settings,
)

In [ ]:
def prepare_sortition_inputs(features_df, people_df, settings, number_people_wanted):
    """Prepare inputs for the sortition algorithm."""
    features_input = features_df.copy()
    features_input.columns = features_input.columns.str.lower()

    features = read_in_features(
        list(features_input.columns),
        features_input.fillna("").to_dict(orient="records"),
        number_people_wanted,
    )[0]

    people = read_in_people(
        list(people_df.columns),
        people_df.fillna("").to_dict(orient="records"),
        features,
        settings,
    )[0]

    return features, people

In [ ]:
settings = Settings(
    id_column=id_column,
    columns_to_keep=columns_to_keep,
    selection_algorithm=selection_algorithm,
)

features, people = prepare_sortition_inputs(
    features_df, people_df, settings, number_people_wanted
)

print(f"Features loaded: {len(features)} categories")
print(f"Candidates: {len(people)}")
print(f"Number to select: {number_people_wanted}")

In [ ]:
success, selected_panels, report = run_stratification(
    features=features,
    people=people,
    number_people_wanted=number_people_wanted,
    settings=settings,
)

print(report.as_text())

if not success:
    if report.last_error():
        print(f"\nError: {report.last_error()}")
    raise RuntimeError("Sortition selection failed. See report above.")

selected_people = selected_panels[0]
print(f"\nSuccessfully selected {len(selected_people)} people")

In [ ]:
selected_panel_df = (
    people_df
    .loc[people_df[id_column].isin(selected_people), [id_column]]
    .reset_index(drop=True)
)
selected_panel_df["SELECTION_TIMESTAMP"] = pd.Timestamp.now("UTC")

print(f"Selected panel shape: {selected_panel_df.shape}")
selected_panel_df.head()

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.use_database(TARGET_DATABASE)
session.use_schema(TARGET_SCHEMA)
snowpark_df = session.create_dataframe(selected_panel_df)

snowpark_df.write.mode("append").save_as_table(
    f"{TARGET_DATABASE}.{TARGET_SCHEMA}.{TARGET_TABLE}",
    column_order="name",
)

print(f"Successfully wrote {len(selected_panel_df)} rows to {TARGET_DATABASE}.{TARGET_SCHEMA}.{TARGET_TABLE}")